# PDPA Assistant — Day 2: Measure accuracy

This notebook gives your project a **real number**: how often retrieval finds the correct page.
It also checks that off-topic questions get "I don't know".

**Before you run it:**

1. Run `pdpa_week1.ipynb` at least once (it saves the embeddings in `pdpa_db`, which this notebook reuses).
2. Open `test_questions.csv` in Excel. For each **normal** question:
    - open the PDF named in `correct_file` and find the page that answers it
    - type that page number in `correct_page` (several pages: separate with `;`, e.g. `12;13`)
    - if the answer is in a different file, fix `correct_file` too
    - if **another document also answers it**, list those pages in `also_correct` as `file:page`,
      e.g. `key_concepts.pdf:143` (several: separate with `;`)
3. Add your own questions if you like. Use `normal` or `off-topic` in the `type` column.
4. Save it as **CSV** (not .xlsx).

**Why find the pages yourself?** The test checks the AI against the truth. If the AI wrote the answer key,
you'd be testing it against itself.

## Settings

Use the **same** model names as the Week 1 notebook — retrieval only works if questions and chunks
are embedded by the same model.

In [ ]:
from pathlib import Path

PROJECT_FOLDER = Path.cwd()
DB_FOLDER = PROJECT_FOLDER / "pdpa_db"
TEST_FILE = PROJECT_FOLDER / "test_questions.csv"
RESULTS_FILE = PROJECT_FOLDER / "test_results.csv"

EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL = "llama3.2"
TOP_K = 4

## Load the vector index saved in Week 1

We **load** the saved embeddings instead of re-embedding every chunk — much faster.

In [ ]:
import json
import numpy as np
import ollama

EMBEDDINGS = np.load(DB_FOLDER / "embeddings.npy")
with open(DB_FOLDER / "chunks.json", encoding="utf-8") as f:
    all_chunks = json.load(f)
print(f"Loaded {len(all_chunks)} chunks")


def embed_texts(texts):
    response = ollama.embed(model=EMBED_MODEL, input=texts)
    return response["embeddings"]


def retrieve(question, top_k=TOP_K):
    q = np.array(embed_texts([question])[0], dtype=np.float32)
    q = q / np.linalg.norm(q)
    scores = EMBEDDINGS @ q
    best = np.argsort(scores)[::-1][:top_k]
    return [{"text": all_chunks[i]["text"], "source": all_chunks[i]["source"],
             "page": all_chunks[i]["page"], "similarity": round(float(scores[i]), 3)} for i in best]

## Load the test questions

The cell warns you about any normal question that still has no page filled in — those are skipped.

In [ ]:
import csv

with open(TEST_FILE, encoding="utf-8-sig") as f:
    tests = list(csv.DictReader(f))

normal = [t for t in tests if t["type"].strip().lower() == "normal"]
off_topic = [t for t in tests if t["type"].strip().lower() == "off-topic"]

print(f"{len(tests)} questions: {len(normal)} normal, {len(off_topic)} off-topic")

missing = [t["question"] for t in normal if not t["correct_page"].strip()]
for q in missing:
    print("  No page filled in yet (will be skipped):", q)

---
## Test 1 — Retrieval accuracy

For each normal question, we retrieve the top chunks and check:
**is the correct file AND page among them?** That's a *hit*.

The percentage of hits is your **retrieval score** (in the field it's called *hit rate at k*, here k = 4).

In [ ]:
def parse_pages(text):
    """'12;13' → {12, 13}"""
    return {int(p) for p in text.replace(",", ";").split(";") if p.strip().isdigit()}


def parse_also(text):
    # 'key_concepts.pdf:143;pdpa_act.pdf:65' -> {('key_concepts.pdf', 143), ('pdpa_act.pdf', 65)}
    pairs = set()
    for item in (text or "").split(";"):
        if ":" in item:
            file, page = item.rsplit(":", 1)
            if page.strip().isdigit():
                pairs.add((file.strip(), int(page)))
    return pairs


results = []
for t in normal:
    pages = parse_pages(t["correct_page"])
    if not pages:
        continue
    correct = {(t["correct_file"].strip(), p) for p in pages} | parse_also(t.get("also_correct", ""))
    found = retrieve(t["question"])
    hit = any((c["source"], c["page"]) in correct for c in found)
    retrieved = "; ".join(f"{c['source']} p{c['page']}" for c in found)
    results.append({"question": t["question"], "type": "normal",
                    "expected": f"{t['correct_file']} p{t['correct_page']}",
                    "retrieved": retrieved, "result": "HIT" if hit else "MISS"})
    print(f"{'HIT ' if hit else 'MISS'}  {t['question']}")
    if not hit:
        print(f"        expected {t['correct_file']} p{t['correct_page']}, got: {retrieved}")

hits = sum(r["result"] == "HIT" for r in results)
if results:
    print(f"\nRETRIEVAL SCORE: {hits} of {len(results)} correct ({hits / len(results):.0%})")
else:
    print("No scored questions yet — fill in correct_page in test_questions.csv")

**Look at every MISS.** Common reasons:
- the answer really is on another page too (update your answer key),
- the question uses very different words from the document,
- the chunk containing the answer is split badly.

Write down 1–2 reasons — these are great interview stories.

---
## Test 2 — Off-topic questions

These need the AI to write an answer, so this step uses the chat model and takes a bit longer.
A **pass** means the assistant said it doesn't have the information instead of making something up.

In [ ]:
REFUSAL = "I don't have information on that in my PDPA documents."


def build_prompt(question, found):
    context = "\n\n".join(f"[{n}] ({c['source']}, page {c['page']})\n{c['text']}"
                           for n, c in enumerate(found, start=1))
    return f"""You are an assistant that explains Singapore's Personal Data Protection Act (PDPA).

Rules:
1. Answer using ONLY the numbered sources below.
2. After each fact, cite the source number in square brackets, for example [1].
3. If the sources do not contain the answer, reply exactly: "{REFUSAL}"
4. Use simple, clear language that a small business owner can understand.

Sources:
{context}

Question: {question}
Answer:"""


def generate(prompt):
    response = ollama.chat(model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}],
                           options={"temperature": 0})
    return response["message"]["content"]


refused = 0
for t in off_topic:
    answer = generate(build_prompt(t["question"], retrieve(t["question"])))
    passed = "don't have information" in answer.lower() or "do not have information" in answer.lower()
    refused += passed
    results.append({"question": t["question"], "type": "off-topic", "expected": "refusal",
                    "retrieved": answer[:150].replace("\n", " "), "result": "PASS" if passed else "FAIL"})
    print(f"{'PASS' if passed else 'FAIL'}  {t['question']}")
    if not passed:
        print("        answer was:", answer[:200])

if off_topic:
    print(f"\nOFF-TOPIC: {refused} of {len(off_topic)} correctly refused")

---
## Save the results

Saves every question and its result to `test_results.csv` — useful for your README and for comparing
before/after when you try improvements later.

In [ ]:
with open(RESULTS_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "type", "expected", "retrieved", "result"])
    writer.writeheader()
    writer.writerows(results)

print(f"Saved {len(results)} results to {RESULTS_FILE}")
if results and normal:
    print(f"\nFor your README and resume: correct page retrieved for {hits} of {hits + sum(r['result'] == 'MISS' for r in results)} test questions.")

## Done for Day 2

Write down:
- your **retrieval score** (e.g. 8 of 10)
- your **off-topic result** (e.g. 2 of 2 refused)
- 1–2 reasons for any misses

Next (Day 3): put the project on GitHub and fill in `README.md`.

**Later upgrade:** change `CHUNK_SIZE` or `TOP_K` in the Week 1 notebook, rebuild the database,
run this notebook again and compare scores. That's your "before vs after" improvement story.